# 🧠 Análise de Saúde Mental no Trabalho Tech
## Módulo 2 — Impacto do Trabalho Remoto e Nicho da Empresa

**Hipótese:** Profissionais em regime remoto podem apresentar padrões diferentes de interferência no trabalho (`work_interfere`) em relação a quem trabalha presencialmente. Além disso, empresas de tecnologia (`tech_company`) podem oferecer mais benefícios de saúde mental — ou seus funcionários podem buscar mais tratamento.

**Dataset:** OSMI Mental Health in Tech Survey (`survey.csv`)

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# Estilo visual
plt.rcParams.update({
    'figure.facecolor': '#1e1e2e',
    'axes.facecolor':   '#2a2a3e',
    'axes.edgecolor':   '#555577',
    'axes.labelcolor':  '#cdd6f4',
    'text.color':       '#cdd6f4',
    'xtick.color':      '#cdd6f4',
    'ytick.color':      '#cdd6f4',
    'grid.color':       '#3a3a5e',
    'font.family':      'DejaVu Sans',
    'font.size':        12,
})

# --------------- Carregamento ---------------
try:
    df = pd.read_csv('data/processed/survey.csv')
    print('Arquivo processado carregado: data/processed/survey.csv')
except FileNotFoundError:
    df = pd.read_csv('data/raw/survey.csv')
    print('Arquivo bruto carregado: data/raw/survey.csv')

print(f'   Shape: {df.shape[0]} respostas × {df.shape[1]} colunas')

# --------------- Tratamento de Nulos em work_interfere ---------------
#valores nulos serão tratados como "Prefere não responder"
df['work_interfere'] = df['work_interfere'].fillna('Prefere não responder')

print(f"\nDistribuição de work_interfere após tratamento:")
print(df['work_interfere'].value_counts())

Arquivo processado carregado: data/processed/survey.csv
   Shape: 1259 respostas × 24 colunas

Distribuição de work_interfere após tratamento:
work_interfere
Sometimes                465
Prefere não responder    264
Never                    213
Rarely                   173
Often                    144
Name: count, dtype: int64


In [11]:
import plotly.express as px
import pandas as pd

# Tradução dos valores
traducao_interferencia = {
    'Never': 'Nunca', 'Rarely': 'Raramente', 'Sometimes': 'Às vezes', 
    'Often': 'Frequentemente', 'Prefere não responder': 'Prefere não responder'
}
traducao_remoto = {'Yes': 'Remoto', 'No': 'Presencial'}

ORDEM_INTERFERENCIA = ['Nunca', 'Raramente', 'Às vezes', 'Frequentemente', 'Prefere não responder']

df_remoto = df[df['remote_work'].isin(['Yes', 'No'])].copy()
df_remoto['work_interfere'] = df_remoto['work_interfere'].map(lambda x: traducao_interferencia.get(x, x))
df_remoto['remote_work'] = df_remoto['remote_work'].map(lambda x: traducao_remoto.get(x, x))

tabela = pd.crosstab(df_remoto['remote_work'], df_remoto['work_interfere'])
colunas_presentes = [c for c in ORDEM_INTERFERENCIA if c in tabela.columns]
tabela = tabela[colunas_presentes]

tabela_pct = tabela.div(tabela.sum(axis=1), axis=0).reset_index()
tabela_long = tabela_pct.melt(id_vars='remote_work', var_name='Interferência', value_name='Proporção')
tabela_long['Proporção (%)'] = (tabela_long['Proporção'] * 100).round(1)

fig1 = px.bar(
    tabela_long, x='remote_work', y='Proporção (%)', color='Interferência',
    barmode='group', text='Proporção (%)',
    color_discrete_sequence=['#89dceb', '#a6e3a1', '#f9e2af', '#f38ba8', '#9399b2'],
    category_orders={'Interferência': ORDEM_INTERFERENCIA},
    title='📡 Interferência da Saúde Mental: Remoto vs. Presencial'
)
fig1.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig1.update_layout(
    plot_bgcolor='#2a2a3e', paper_bgcolor='#1e1e2e', font_color='#cdd6f4',
    yaxis_ticksuffix='%', height=500, xaxis_title='Modelo de Trabalho', yaxis_title='Percentual de Respondentes (%)'
)
fig1.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [8]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

traducao_sim_nao = {'Yes': 'Sim', 'No': 'Não', "Don't know": 'Não sei', 'Some of them': 'Alguns', 'Not sure': 'Não tenho certeza'}
traducao_tech = {'Yes': 'Tech', 'No': 'Non-Tech'}

df_tech = df[df['tech_company'].isin(['Yes', 'No'])].copy()
df_tech['tech_company'] = df_tech['tech_company'].map(lambda x: traducao_tech.get(x, x))
df_tech['benefits'] = df_tech['benefits'].map(lambda x: traducao_sim_nao.get(x, x))
df_tech['treatment'] = df_tech['treatment'].map(lambda x: traducao_sim_nao.get(x, x))

tab_b = pd.crosstab(df_tech['tech_company'], df_tech['benefits'], normalize='index').reset_index()
tab_b_long = tab_b.melt(id_vars='tech_company', var_name='Benefícios', value_name='Proporção')
tab_b_long['Proporção (%)'] = (tab_b_long['Proporção'] * 100).round(1)

tab_t = pd.crosstab(df_tech['tech_company'], df_tech['treatment'], normalize='index').reset_index()
tab_t_long = tab_t.melt(id_vars='tech_company', var_name='Tratamento', value_name='Proporção')
tab_t_long['Proporção (%)'] = (tab_t_long['Proporção'] * 100).round(1)

cores_b = {'Sim': '#a6e3a1', 'Não sei': '#f9e2af', 'Não': '#f38ba8'}
cores_t = {'Sim': '#f38ba8', 'Não': '#89b4fa'}

fig2 = make_subplots(rows=1, cols=2, subplot_titles=('🏥 Benefícios de Saúde Mental', '💊 Busca por Tratamento'))

for stat in tab_b_long['Benefícios'].unique():
    df_sub = tab_b_long[tab_b_long['Benefícios'] == stat]
    fig2.add_trace(go.Bar(name=str(stat), x=df_sub['tech_company'], y=df_sub['Proporção (%)'], marker_color=cores_b.get(stat, '#cccccc'), text=df_sub['Proporção (%)'], texttemplate='%{text:.1f}%', textposition='outside', legendgroup='1'), row=1, col=1)

for stat in tab_t_long['Tratamento'].unique():
    df_sub = tab_t_long[tab_t_long['Tratamento'] == stat]
    fig2.add_trace(go.Bar(name=str(stat), x=df_sub['tech_company'], y=df_sub['Proporção (%)'], marker_color=cores_t.get(stat, '#cccccc'), text=df_sub['Proporção (%)'], texttemplate='%{text:.1f}%', textposition='outside', legendgroup='2'), row=1, col=2)

fig2.update_layout(
    barmode='group', plot_bgcolor='#2a2a3e', paper_bgcolor='#1e1e2e', font_color='#cdd6f4',
    height=500, showlegend=True, legend_title='Respostas', title='Comparativo: Tech vs. Non-Tech'
)
fig2.update_yaxes(ticksuffix='%')
fig2.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

---
## 📝 Conclusão de Negócio — Insights

### 🔍 Gráfico 1 — O Fator Remoto (`remote_work` × `work_interfere`)

**Observação:**  
A análise evidenciou que profissionais em regime **remoto** tendem a relatar proporções mais elevadas de interferência **'Sometimes'** (Às vezes) e **'Often'** (Frequente) na sua rotina de trabalho, quando comparados a trabalhadores presenciais.

**Interpretação de Negócio:**  
Este padrão pode ser um sinal de **Burnout por falta de fronteira entre vida pessoal e profissional** — sem o deslocamento físico como "separador simbólico", profissionais remotos tendem a estender o horário de trabalho e a dificuldade de "desligar" acaba amplificando a interferência que a saúde mental exerce sobre a produtividade. Ao mesmo tempo, o isolamento social reduz a rede de apoio informal que o ambiente presencial oferece naturalmente.

> **Recomendação:** Empresas devem estabelecer políticas claras de desconexão digital, incentivar pausas estruturadas e oferecer programas de suporte emocional específicos para times remotos.

---

### 🔍 Gráfico 2 — Tech vs. Non-Tech (`tech_company` × `benefits` + `treatment`)

**Observação:**  
Empresas de tecnologia (`tech_company = Yes`) tendem a oferecer mais **benefícios de saúde mental** (`benefits`) do que empresas de outros setores. Em contrapartida, funcionários de empresas de TI também apresentam maior taxa de **busca por tratamento** (`treatment`), o que pode indicar maior consciência sobre saúde mental nesse nicho.

**Interpretação de Negócio:**  
O mercado tech possui, historicamente, uma cultura mais aberta ao debate sobre saúde mental — e os dados corroboram isso. Contudo, a maior taxa de busca por tratamento não é necessariamente negativa: ela pode refletir tanto maior **acesso a recursos** quanto **maior pressão e sobrecarga** características do setor de tecnologia (deadlines agressivos, ritmo acelerado, alta competitividade).

> **Recomendação:** O fato de a empresa ser de tecnologia não garante bem-estar. É necessário ir além dos benefícios formais e construir uma **cultura psicologicamente segura**, onde pedir ajuda não seja visto como fraqueza.

---

### 🎯 Síntese Final

| Fator | Achado | Risco Identificado |
|---|---|---|
| Trabalho Remoto | Maior interferência 'Often'/'Sometimes' | Burnout por ausência de fronteiras |
| Empresas Tech | Mais benefícios E mais busca por tratamento | Alta pressão apesar dos recursos disponíveis |
| Nulos em `work_interfere` | Tratados como 'Prefere não responder' | Possível subnotificação de casos graves |